In [ ]:
import requests

url = "https://ndownloader.figshare.com/files/38030910"
response = requests.get(url, stream=True, timeout=60)

print("Status:", response.status_code)
print("Content type:", response.headers.get("content-type"))
print("File size:", response.headers.get("content-length"))

In [ ]:
import requests

article_id = 21431547
api_url = f"https://api.figshare.com/v2/articles/{article_id}"

response = requests.get(api_url, timeout=30)
print("API status:", response.status_code)
response.raise_for_status()

article = response.json()

for file in article.get("files", []):
    print(
        "Name:", file["name"],
        "| Size (MB):", round(file["size"] / (1024 * 1024), 1),
        "| Download:", file["download_url"]
    )

## Next: YOLOv7 training
This notebook only prepares data. Add the YOLOv7 framework and compatible pretrained weights as Kaggle inputs or download them separately. Copy this bundle's custom model YAML to the framework's `cfg/training/` and the hyperparameter YAML to its `data/`. Train using the generated `/kaggle/working/RDD2022_India_ChinaDrone_YOLO/India_ChinaDrone.yaml`. Check the upstream `train.py -h` output for exact arguments before training.

In [ ]:
import requests

url = "https://ndownloader.figshare.com/files/38030910"

r = requests.get(url, headers={"Range": "bytes=0-1023"},
                 stream=True, timeout=30)

print("Status:", r.status_code)
print("Content-Range:", r.headers.get("Content-Range"))
print("Accept-Ranges:", r.headers.get("Accept-Ranges"))

r.close()

In [ ]:
!pip -q install remotezip

from remotezip import RemoteZip

url = "https://ndownloader.figshare.com/files/38030910"

with RemoteZip(url) as z:
    names = z.namelist()
    matches = [
        name for name in names
        if "india" in name.lower()
        or "china_drone" in name.lower()
    ]
    for name in matches:
        info = z.getinfo(name)
        print(f"{name}  |  {info.file_size / (1024**2):.1f} MB")

In [ ]:
%cd /kaggle/working
!git clone https://github.com/WongKinYiu/yolov7.git

In [ ]:
from remotezip import RemoteZip
import os

url = "https://ndownloader.figshare.com/files/38030910"
output_dir = "/kaggle/working/RDD2022_selected"
os.makedirs(output_dir, exist_ok=True)

with RemoteZip(url) as z:
    for member in ["RDD2022/India.zip", "RDD2022/China_Drone.zip"]:
        output_path = os.path.join(output_dir, os.path.basename(member))

        print(f"Downloading {member}...")
        with z.open(member) as src, open(output_path, "wb") as dst:
            while True:
                chunk = src.read(8 * 1024 * 1024)
                if not chunk:
                    break
                dst.write(chunk)

        print(f"Saved: {output_path} ({os.path.getsize(output_path) / (1024**2):.1f} MB)")

print("Both country archives downloaded.")

In [ ]:
import zipfile, os

base = "/kaggle/working/RDD2022_selected"

for name in ["India", "China_Drone"]:
    zip_path = f"{base}/{name}.zip"
    out = f"{base}/{name}"
    os.makedirs(out, exist_ok=True)

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out)

    print(name, "extracted")

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/RDD2022_selected")

for country in ["India", "China_Drone"]:
    root = base / country
    print(f"\n{country}:")
    for p in root.rglob("images"):
        print("Images:", p)
    for p in root.rglob("xmls"):
        print("Annotations:", p)

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/RDD2022_selected")

for country in ["India", "China_Drone"]:
    root = base / country
    images = [p for p in root.rglob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    xmls = list(root.rglob("*.xml"))
    print(f"{country}: {len(images)} images, {len(xmls)} XML annotations")

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/RDD2022_selected")

for country in ["India", "China_Drone"]:
    root = base / country
    train = next(root.rglob("train/images"))
    xml_dir = train.parent / "annotations" / "xmls"

    images = list(train.glob("*"))
    images = [p for p in images if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    xmls = list(xml_dir.glob("*.xml"))

    image_stems = {p.stem for p in images}
    xml_stems = {p.stem for p in xmls}

    print(f"\n{country}")
    print("Training images:", len(images))
    print("XML annotations:", len(xmls))
    print("Images without XML:", len(image_stems - xml_stems))
    print("XML without images:", len(xml_stems - image_stems))

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil, random

BASE = Path("/kaggle/working/RDD2022_selected")
OUT = Path("/kaggle/working/RDD2022_YOLO")
random.seed(42)

# Standard RDD2022 road-damage classes
CLASSES = ["D00", "D10", "D20", "D40"]
class_to_id = {name: i for i, name in enumerate(CLASSES)}

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

def find_annotations(image, xml_dir):
    return xml_dir / f"{image.stem}.xml"

def convert_xml(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    width, height = int(size.find("width").text), int(size.find("height").text)

    rows = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_to_id:
            continue
        box = obj.find("bndbox")
        xmin, ymin = float(box.find("xmin").text), float(box.find("ymin").text)
        xmax, ymax = float(box.find("xmax").text), float(box.find("ymax").text)

        x = ((xmin + xmax) / 2) / width
        y = ((ymin + ymax) / 2) / height
        w = (xmax - xmin) / width
        h = (ymax - ymin) / height

        rows.append(f"{class_to_id[name]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.write_text("\n".join(rows))

for country in ["India", "China_Drone"]:
    root = BASE / country
    train_img_dir = next(root.rglob("train/images"))
    xml_dir = train_img_dir.parent / "annotations" / "xmls"

    images = sorted(
        p for p in train_img_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    )

    train_images, val_images = train_test_split(
        images, test_size=0.20, random_state=42
    )

    for split, subset in [("train", train_images), ("val", val_images)]:
        for img in subset:
            # Prefix filenames to avoid collisions across countries
            new_name = f"{country}_{img.name}"
            shutil.copy2(img, OUT / "images" / split / new_name)
            convert_xml(
                find_annotations(img, xml_dir),
                OUT / "labels" / split / f"{Path(new_name).stem}.txt"
            )

    # Preserve India's official test images as images only;
    # official test annotations can be added later if needed.
    if country == "India":
        test_img_dir = next(root.rglob("test/images"))
        for img in test_img_dir.iterdir():
            if img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                shutil.copy2(
                    img, OUT / "images" / "test" / f"India_{img.name}"
                )

    print(country, "train:", len(train_images), "val:", len(val_images))

print("\nConversion finished:", OUT)

In [ ]:
from pathlib import Path
from collections import Counter

OUT = Path("/kaggle/working/RDD2022_YOLO")

for split in ["train", "val"]:
    images = list((OUT / "images" / split).glob("*"))
    labels = list((OUT / "labels" / split).glob("*.txt"))

    class_counts = Counter()
    empty_labels = 0

    for label in labels:
        lines = label.read_text().strip().splitlines()
        if not lines:
            empty_labels += 1
        for line in lines:
            class_counts[line.split()[0]] += 1

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Empty label files:", empty_labels)
    print("Objects by class:", dict(class_counts))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

BASE = Path("/kaggle/working/RDD2022_selected")

for country in ["India", "China_Drone"]:
    root = BASE / country
    xml_dir = next(root.rglob("train/annotations/xmls"))

    classes = Counter()
    no_objects = 0

    for xml_file in xml_dir.glob("*.xml"):
        tree = ET.parse(xml_file)
        objects = tree.findall(".//object")

        if not objects:
            no_objects += 1

        for obj in objects:
            name = obj.findtext("name", default="UNKNOWN").strip()
            classes[name] += 1

    print(f"\n{country}")
    print("Object classes:", dict(classes))
    print("XMLs without objects:", no_objects)

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

BASE = Path("/kaggle/working/RDD2022_selected")

targets = {
    "India": ["D44", "D01", "D11", "D50", "D0w0"],
    "China_Drone": ["Repair", "Block crack"]
}

for country, wanted in targets.items():
    xml_dir = next((BASE / country).rglob("train/annotations/xmls"))
    print(f"\n--- {country} ---")

    shown = set()
    for xml_file in xml_dir.glob("*.xml"):
        root = ET.parse(xml_file).getroot()

        for obj in root.findall("object"):
            name = obj.findtext("name", "").strip()
            if name in wanted and name not in shown:
                print(f"\nClass: {name}")
                print("XML:", xml_file.name)
                print(ET.tostring(obj, encoding="unicode"))
                shown.add(name)

        if len(shown) == len(wanted):
            break

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/RDD2022_selected")

for p in BASE.rglob("*"):
    if p.is_file() and p.name.lower() in [
        "label_map.pbtxt",
        "readme.txt",
        "directory_structure_crddc_rdd2022.txt"
    ]:
        print(f"\n--- {p} ---")
        try:
            print(p.read_text(errors="ignore")[:12000])
        except Exception as e:
            print("Could not read:", e)

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil
from collections import Counter

BASE = Path("/kaggle/working/RDD2022_selected")
OUT = Path("/kaggle/working/RDD2022_YOLO")

# Standard four-class RDD2022 task
CLASSES = ["D00", "D10", "D20", "D40"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

# Recreate output folders to prevent stale files from earlier conversions
if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

def convert_xml(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")

    width = int(size.findtext("width"))
    height = int(size.findtext("height"))
    rows = []

    for obj in root.findall("object"):
        name = obj.findtext("name", "").strip()

        # Skip classes outside this four-class experiment
        if name not in CLASS_TO_ID:
            continue

        box = obj.find("bndbox")
        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        # Clip coordinates to image bounds
        xmin = max(0, min(xmin, width))
        xmax = max(0, min(xmax, width))
        ymin = max(0, min(ymin, height))
        ymax = max(0, min(ymax, height))

        if xmax <= xmin or ymax <= ymin:
            continue

        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        box_width = (xmax - xmin) / width
        box_height = (ymax - ymin) / height

        rows.append(
            f"{CLASS_TO_ID[name]} "
            f"{x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}"
        )

    label_path.write_text("\n".join(rows))

def get_image_dir(country, split):
    root = BASE / country
    matches = list(root.rglob(f"{split}/images"))
    if not matches:
        raise FileNotFoundError(f"Missing {split}/images for {country}")
    return matches[0]

# Split each source separately to preserve both viewpoints in train and validation
for country in ["India", "China_Drone"]:
    image_dir = get_image_dir(country, "train")
    xml_dir = image_dir.parent / "annotations" / "xmls"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    # Verify every image has a matching XML
    missing = [p.name for p in images if not (xml_dir / f"{p.stem}.xml").exists()]
    if missing:
        raise FileNotFoundError(
            f"{country}: {len(missing)} images lack XML annotations. "
            f"Examples: {missing[:5]}"
        )

    train_images, val_images = train_test_split(
        images, test_size=0.20, random_state=42
    )

    for split, subset in [("train", train_images), ("val", val_images)]:
        for img in subset:
            new_name = f"{country}_{img.name}"
            shutil.copy2(img, OUT / "images" / split / new_name)

            xml_path = xml_dir / f"{img.stem}.xml"
            label_path = OUT / "labels" / split / f"{Path(new_name).stem}.txt"
            convert_xml(xml_path, label_path)

    print(f"{country}: {len(train_images)} train, {len(val_images)} val")

# Preserve India's official test images without mixing them into train/val
india_test_dir = get_image_dir("India", "test")
for img in india_test_dir.iterdir():
    if img.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        new_name = f"India_{img.name}"
        shutil.copy2(img, OUT / "images" / "test" / new_name)

print("\nConversion complete:", OUT)

In [ ]:
from pathlib import Path

OUT = Path("/kaggle/working/RDD2022_YOLO")

for split in ["train", "val", "test"]:
    for kind in ["images", "labels"]:
        p = OUT / kind / split
        count = len(list(p.iterdir())) if p.exists() else 0
        print(f"{kind}/{split}: {count} files")

In [ ]:
from pathlib import Path
import yaml

OUT = Path("/kaggle/working/RDD2022_YOLO")

data = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

yaml_path = OUT / "rdd2022.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(yaml_path.read_text())

In [ ]:
import sys, torch, numpy

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
%cd /kaggle/working/yolov7

!grep -v -E '^(numpy|torch|torchvision)([<=>!~].*)?$' requirements.txt > /tmp/yolov7-requirements.txt
!pip install -r /tmp/yolov7-requirements.txt

In [ ]:
import sys, torch, numpy, cv2, yaml
import google.protobuf

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("OpenCV:", cv2.__version__)
print("Protobuf:", google.protobuf.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /kaggle/working/yolov7
!python train.py --help

In [ ]:
import sys, torch, numpy, cv2, yaml
import google.protobuf

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("Protobuf:", google.protobuf.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /kaggle/working/yolov7
!python train.py --help

In [ ]:
%cd /kaggle/working/yolov7
!wget -O yolov7_training.pt https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt

In [ ]:
from pathlib import Path
import yaml

p = Path("/kaggle/working/RDD2022_YOLO")
data = yaml.safe_load(open(p / "rdd2022.yaml"))

print(data)
print("Train images:", len(list((p / "images/train").glob("*"))))
print("Validation images:", len(list((p / "images/val").glob("*"))))
print("Weights downloaded:", Path("/kaggle/working/yolov7/yolov7_training.pt").exists())

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 8 \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7_training.pt \
  --name rdd2022_india_china_drone \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 50

In [ ]:
%cd /kaggle/working/yolov7

import torch

original_load = torch.load

def compatible_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return original_load(*args, **kwargs)

torch.load = compatible_load

print("Compatibility patch applied.")

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 8 \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7_training.pt \
  --name rdd2022_india_china_drone \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 50

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("train.py")
s = p.read_text()
old = "torch.load(weights, map_location=device)"
new = "torch.load(weights, map_location=device, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("Patched train.py successfully.")
elif new in s:
    print("Patch already applied.")
else:
    print("Could not find the expected line. Please share the relevant code around line 71.")

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 8 \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7_training.pt \
  --name rdd2022_india_china_drone \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 50

In [ ]:
from pathlib import Path
import yaml

p = Path("/kaggle/working/RDD2022_YOLO/rdd2022.yaml")
print(p.read_text())
print("Validation folder exists:",
      Path("/kaggle/working/RDD2022_YOLO/images/val").exists())

In [ ]:
from pathlib import Path
import yaml

out = Path("/kaggle/working/RDD2022_YOLO")

data = {
    "train": str(out / "images/train"),
    "val": str(out / "images/val"),
    "test": str(out / "images/test"),
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"]
}

with open(out / "rdd2022.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print((out / "rdd2022.yaml").read_text())

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 8 \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7_training.pt \
  --name rdd2022_india_china_drone \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 50

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4/weights")

print("Weights folder exists:", p.exists())
if p.exists():
    for f in p.iterdir():
        print(f.name, f"{f.stat().st_size / (1024**2):.1f} MB")

In [ ]:
import shutil

src = "/kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4/weights/best.pt"
dst = "/kaggle/working/best_road_damage_model.pt"

shutil.copy2(src, dst)
print("Ready:", dst)

In [ ]:
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/best_road_damage_model.pt"))

In [ ]:
import os

p = "/kaggle/working/best_road_damage_model.pt"
print("Exists:", os.path.exists(p))
print("Size (MB):", round(os.path.getsize(p) / (1024**2), 2)
      if os.path.exists(p) else "File missing")

In [ ]:
/kaggle/working/best_road_damage_model.pt

In [ ]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/best_road_damage_model.pt'))

In [ ]:
import zipfile
from pathlib import Path

model = Path("/kaggle/working/best_road_damage_model.pt")
archive = Path("/kaggle/working/road_damage_model.zip")

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(model, arcname=model.name)

print("ZIP created:", archive)
print("Size (MB):", round(archive.stat().st_size / (1024**2), 2))

In [ ]:
from pathlib import Path

model = Path(
    "/kaggle/working/yolov7/runs/train/"
    "rdd2022_india_china_drone4/weights/best.pt"
)

print("Model exists:", model.exists())

In [ ]:
from pathlib import Path

val_dir = Path("/kaggle/working/RDD2022_YOLO/images/val")
images = list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png"))

print("Available images:", len(images))
print("Sample image:", images[0] if images else "No images found")

In [ ]:
%cd /kaggle/working/yolov7

!python detect.py \
  --weights /kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4/weights/best.pt \
  --source /kaggle/working/RDD2022_YOLO/images/val \
  --img-size 640 \
  --conf-thres 0.25 \
  --device 0 \
  --name road_damage_test

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("models/experimental.py")
s = p.read_text()

old = "torch.load(w, map_location=map_location)"
new = "torch.load(w, map_location=map_location, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("Detection compatibility fix applied.")
elif new in s:
    print("Fix already applied.")
else:
    print("Expected line not found; please share the code around line 252.")

In [ ]:
from pathlib import Path

val_dir = Path("/kaggle/working/RDD2022_YOLO/images/val")
image = next(
    p for p in val_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print("Testing:", image)

!python detect.py \
  --weights /kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4/weights/best.pt \
  --source "{image}" \
  --img-size 640 \
  --conf-thres 0.25 \
  --device 0 \
  --name road_damage_test

In [ ]:
from IPython.display import display, Image
from pathlib import Path

result = Path(
    "/kaggle/working/yolov7/runs/detect/"
    "road_damage_test2/India_India_007706.jpg"
)

display(Image(filename=str(result)))

In [ ]:
%cd /kaggle/working/yolov7

!python detect.py \
  --weights /kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4/weights/best.pt \
  --source /kaggle/working/RDD2022_YOLO/images/val/India_India_007706.jpg \
  --img-size 640 \
  --conf-thres 0.05 \
  --device 0 \
  --name low_conf_test

In [ ]:
from IPython.display import display, Image
display(Image(filename="/kaggle/working/yolov7/runs/detect/low_conf_test/India_India_007706.jpg"))

In [ ]:
from pathlib import Path

run = Path(
    "/kaggle/working/yolov7/runs/train/"
    "rdd2022_india_china_drone4"
)

for f in run.iterdir():
    if f.suffix.lower() in {".png", ".csv"}:
        print(f.name)

In [ ]:
from IPython.display import display, Image

display(Image(
    filename="/kaggle/working/yolov7/runs/train/"
    "rdd2022_india_china_drone4/results.png"
))

In [ ]:
from pathlib import Path

weights = Path(
    "/kaggle/working/yolov7/runs/train/"
    "rdd2022_india_china_drone4/weights"
)

for name in ["best.pt", "last.pt"]:
    p = weights / name
    print(name, "exists:", p.exists(),
          "size (MB):", round(p.stat().st_size / 1024**2, 1)
          if p.exists() else "-")

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("utils/datasets.py")
s = p.read_text()

old = "torch.load(cache_path)"
new = "torch.load(cache_path, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("Dataset cache compatibility fixed.")
else:
    print("Checking whether the fix is already applied:", new in s)

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --weights ./runs/train/rdd2022_india_china_drone4/weights/last.pt \
  --resume

In [ ]:
import os

print("Working directory:", os.listdir("/kaggle/working"))
print("Input files:", os.listdir("/kaggle/input"))

In [ ]:
import os
import zipfile

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".zip"):
            print("Found ZIP:", os.path.join(root, file))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", files[:10])

In [ ]:
import os
import shutil

source = "/kaggle/input/datasets/saksh01/trained-dataset/best_road_damage_model.pt"
destination = "/kaggle/working/best_road_damage_model.pt"

shutil.copy2(source, destination)

print("Model copied:", os.path.exists(destination))
print("Size:", round(os.path.getsize(destination) / (1024**2), 2), "MB")

In [ ]:
%cd /kaggle/working
!git clone https://github.com/WongKinYiu/yolov7.git

In [ ]:
import os

print("YOLOv7 folder exists:", os.path.isdir("/kaggle/working/yolov7"))
print("Model exists:", os.path.isfile("/kaggle/working/best_road_damage_model.pt"))
print("Model size (MB):", round(
    os.path.getsize("/kaggle/working/best_road_damage_model.pt") / (1024 * 1024), 2
))

In [ ]:
%cd /kaggle/working/yolov7

!pip install -q thop

In [ ]:
import os

model = "/kaggle/working/best_road_damage_model.pt"
repo = "/kaggle/working/yolov7"

print("Model:", os.path.exists(model))
print("YOLOv7:", os.path.exists(repo))
print("Config:", os.path.exists(repo + "/cfg/training/yolov7.yaml"))
print("Detection script:", os.path.exists(repo + "/detect.py"))

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("models/experimental.py")
s = p.read_text()

s = s.replace(
    "torch.load(w, map_location=map_location)",
    "torch.load(w, map_location=map_location, weights_only=False)"
)

p.write_text(s)
print("Checkpoint-loading compatibility patch applied.")

In [ ]:
import os

extensions = (".jpg", ".jpeg", ".png", ".bmp")
images = []

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(extensions):
            images.append(os.path.join(root, f))

print("Images found:", len(images))
print(*images[:10], sep="\n")

In [ ]:
!wget -O /kaggle/working/test_road.jpg \
"https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg"

In [ ]:
%cd /kaggle/working/yolov7

!python detect.py \
  --weights /kaggle/working/best_road_damage_model.pt \
  --source /kaggle/working/test_road.jpg \
  --img-size 640 \
  --conf-thres 0.05 \
  --device 0

In [ ]:
!wget -O /kaggle/working/road_test.jpg \
"https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg"

In [ ]:
import glob, os

images = glob.glob(
    "/kaggle/working/RDD2022_YOLO/images/val/*"
    "*.jpg"
)

if not images:
    images = glob.glob(
        "/kaggle/working/RDD2022_YOLO/images/train/*"
        "*.jpg"
    )

print("Images found:", len(images))
print("Test image:", images[0] if images else "NONE")

In [ ]:
%cd /kaggle/working/yolov7

!python detect.py \
  --weights /kaggle/working/best_road_damage_model.pt \
  --source "{images[0]}" \
  --img-size 640 \
  --conf-thres 0.15 \
  --device 0

In [ ]:
from IPython.display import display, Image
import glob

outputs = sorted(
    glob.glob("/kaggle/working/yolov7/runs/detect/exp*/*.jpg"),
    key=os.path.getmtime
)

if outputs:
    display(Image(filename=outputs[-1]))
else:
    print("No output image found.")

In [ ]:
from pathlib import Path
import yaml

yaml_path = Path("/kaggle/working/RDD2022_YOLO/rdd2022.yaml")
data = yaml.safe_load(yaml_path.read_text())

root = Path("/kaggle/working/RDD2022_YOLO")

for split in ("train", "val", "test"):
    path = root / data[split]
    print(f"{split}: {path} | exists: {path.exists()}")

data["path"] = str(root)
for split in ("train", "val", "test"):
    data[split] = str(root / data[split])

yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print("\nYAML paths updated.")

In [ ]:
%cd /kaggle/working/yolov7

!python test.py \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 \
  --batch-size 8 \
  --weights /kaggle/working/best_road_damage_model.pt \
  --task val \
  --device 0

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/yolov7")

for folder in ["runs/train", "runs/test", "cfg/training"]:
    p = root / folder
    print(f"\n{folder}: {'FOUND' if p.exists() else 'MISSING'}")
    if p.exists():
        print([x.name for x in list(p.iterdir())[:10]])

print("\nSaved model:", Path("/kaggle/working/best_road_damage_model.pt").exists())

In [ ]:
import torch

p = "/kaggle/working/best_road_damage_model.pt"
ckpt = torch.load(p, map_location="cpu", weights_only=False)

print("Checkpoint type:", type(ckpt).__name__)
if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys()))
    print("Epoch:", ckpt.get("epoch", "Not saved"))
    print("Optimizer available:", ckpt.get("optimizer") is not None)

In [ ]:
from pathlib import Path
import yaml

run = Path("/kaggle/working/yolov7/runs/train/rdd2022_india_china_drone4")
print("Run directory exists:", run.exists())

In [ ]:
from pathlib import Path
import yaml

root = Path("/kaggle/working/RDD2022_YOLO")
cfg = Path("/kaggle/working/yolov7/cfg/training/yolov7.yaml")
weights = Path("/kaggle/working/best_road_damage_model.pt")

print("Train images:", (root / "images/train").exists())
print("Validation images:", (root / "images/val").exists())
print("Training config:", cfg.exists())
print("Saved checkpoint:", weights.exists())

data = yaml.safe_load((root / "rdd2022.yaml").read_text())
print("Classes:", data["names"])
print("Number of classes:", data["nc"])

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("train.py")
s = p.read_text()

old = "torch.load(weights, map_location=device)"
new = "torch.load(weights, map_location=device, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("Patch applied.")
else:
    print("Pattern not found; checking current line:")
    for line in s.splitlines():
        if "torch.load(weights" in line:
            print(line)

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

p = Path("utils/datasets.py")
s = p.read_text()

old = "torch.load(cache_path)"
new = "torch.load(cache_path, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("Fixed cached-label loading.")
else:
    print("Pattern not found; inspect the torch.load line:")
    for line in s.splitlines():
        if "torch.load(cache_path" in line:
            print(line)

In [ ]:
%cd /kaggle/working/yolov7

!python train.py \
  --workers 2 \
  --device 0 \
  --batch-size 8 \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights /kaggle/working/best_road_damage_model.pt \
  --name rdd2022_finetune \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 25

In [ ]:
from pathlib import Path

weights = Path("/kaggle/working/yolov7/runs/train/rdd2022_finetune2/weights")

print("Weights folder exists:", weights.exists())
if weights.exists():
    for p in weights.glob("*.pt"):
        print(p.name, round(p.stat().st_size / (1024**2), 1), "MB")

In [ ]:
from pathlib import Path

for base in [
    Path("/kaggle/working/yolov7/runs/train"),
    Path("/kaggle/working/yolov7/runs"),
    Path("/kaggle/working"),
]:
    print(f"\nSearching: {base}")
    if base.exists():
        found = list(base.glob("**/*.pt"))
        for p in found[:30]:
            print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")
        if not found:
            print("No .pt files found here.")
    else:
        print("Folder missing.")

In [ ]:
import shutil
from pathlib import Path

source = Path(
    "/kaggle/working/yolov7/runs/train/"
    "rdd2022_finetune3/weights/best.pt"
)
destination = Path("/kaggle/working/improved_road_damage_model.pt")

shutil.copy2(source, destination)

print("Backup exists:", destination.exists())
print("Size:", round(destination.stat().st_size / (1024**2), 1), "MB")

In [ ]:
from pathlib import Path

for name in [
    "improved_road_damage_model.pt",
    "best_road_damage_model.pt",
]:
    p = Path("/kaggle/working") / name
    print(name, "EXISTS" if p.exists() else "MISSING")

In [ ]:
%cd /kaggle/working/yolov7

!python test.py \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 \
  --batch-size 8 \
  --weights /kaggle/working/yolov7/runs/train/rdd2022_finetune3/weights/best.pt \
  --task val \
  --device 0

In [ ]:
import random
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

root = Path("/kaggle/working/RDD2022_YOLO")
images = list((root / "images/val").glob("*.jpg"))
random.seed(42)

for img_path in random.sample(images, min(3, len(images))):
    label_path = root / "labels/val" / (img_path.stem + ".txt")
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, x, y, w, h = map(float, parts)
            W, H = img.size
            x1, y1 = (x-w/2)*W, (y-h/2)*H
            x2, y2 = (x+w/2)*W, (y+h/2)*H
            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            draw.text((x1, y1), ["D00","D10","D20","D40"][int(cls)], fill="yellow")

    display(img.resize((640, int(img.height * 640 / img.width))))

In [ ]:
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/RDD2022_YOLO")
labels = list((root / "labels/val").glob("*.txt"))

counts = Counter()
empty = 0
total_boxes = 0

for p in labels:
    lines = [line for line in p.read_text().splitlines() if line.strip()]
    if not lines:
        empty += 1
    for line in lines:
        cls = int(line.split()[0])
        counts[cls] += 1
        total_boxes += 1

print("Label files:", len(labels))
print("Empty label files:", empty)
print("Total boxes:", total_boxes)
print("Boxes by class:", {
    ["D00", "D10", "D20", "D40"][k]: v
    for k, v in sorted(counts.items())
})

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/RDD2022_selected")

print("Original dataset folder exists:", root.exists())

if root.exists():
    for p in list(root.rglob("*.xml"))[:10]:
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/RDD2022_selected")

for p in root.rglob("*"):
    if p.is_dir() and ("annot" in p.name.lower() or "xml" in p.name.lower()):
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/RDD2022_selected/India/India")

for p in root.rglob("*"):
    if p.is_dir() and (
        "test" in p.name.lower()
        or "val" in p.name.lower()
        or "annot" in p.name.lower()
        or "xml" in p.name.lower()
    ):
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/RDD2022_selected/India/India/test")

for p in root.iterdir():
    print(p.name, "— folder" if p.is_dir() else "— file")
    if p.is_dir():
        print("  ", [x.name for x in list(p.iterdir())[:5]])

In [ ]:
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/RDD2022_YOLO/labels/train")
counts = Counter()
empty = 0
invalid = 0
total = 0

for p in root.glob("*.txt"):
    lines = [x for x in p.read_text().splitlines() if x.strip()]
    if not lines:
        empty += 1

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            invalid += 1
            continue

        cls, x, y, w, h = map(float, parts)
        total += 1
        counts[int(cls)] += 1

        if not (0 <= cls <= 3 and 0 <= x <= 1 and
                0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            invalid += 1

print("Label files:", len(list(root.glob("*.txt"))))
print("Empty files:", empty)
print("Total boxes:", total)
print("Boxes by class:", dict(counts))
print("Invalid annotations:", invalid)

In [ ]:
%cd /kaggle/working/yolov7

from pathlib import Path

images = list(Path(
    "/kaggle/working/RDD2022_YOLO/images/val"
).glob("*.jpg"))

print("Validation images:", len(images))
print("Example:", images[0] if images else "No images found")

In [ ]:
img = str(images[0])

!python detect.py --weights runs/train/rdd2022_finetune3/weights/best.pt --source "$img" --img-size 640 --conf-thres 0.10 --device 0 --name compare640

!python detect.py --weights runs/train/rdd2022_finetune3/weights/best.pt --source "$img" --img-size 960 --conf-thres 0.10 --device 0 --name compare960

In [ ]:
from pathlib import Path
from IPython.display import display, Image

base = Path("/kaggle/working/yolov7/runs/detect")

for name in ["compare640", "compare960"]:
    folders = sorted(
        base.glob(name + "*"),
        key=lambda p: p.stat().st_mtime
    )

    if folders:
        images = list(folders[-1].glob("*.jpg"))
        if images:
            print(name)
            display(Image(filename=str(images[0])))
        else:
            print(name, "- no output image found")
    else:
        print(name, "- output folder not found")

In [ ]:
%cd /kaggle/working/yolov7

!python test.py \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 640 \
  --batch-size 8 \
  --weights runs/train/rdd2022_finetune3/weights/best.pt \
  --task val --device 0 --name eval640

!python test.py \
  --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml \
  --img-size 960 \
  --batch-size 4 \
  --weights runs/train/rdd2022_finetune3/weights/best.pt \
  --task val --device 0 --name eval960

In [ ]:
from pathlib import Path
import yaml

run = Path("/kaggle/working/yolov7/runs/train/rdd2022_finetune3")
hyp_path = run / "hyp.yaml"

print("Run exists:", run.exists())
print("Hyperparameter file exists:", hyp_path.exists())

if hyp_path.exists():
    print(yaml.safe_load(hyp_path.read_text()))

In [ ]:
from pathlib import Path

run = Path("/kaggle/working/yolov7/runs/train/rdd2022_finetune3")
for name in ["results.txt", "results.csv"]:
    p = run / name
    print(f"\n{name}:")
    if p.exists():
        print(p.read_text()[-4000:])
    else:
        print("Not found")

In [ ]:
from pathlib import Path
import shutil

src = Path("/kaggle/working/yolov7/runs/train/rdd2022_finetune3/weights/best.pt")
dst = Path("/kaggle/working/baseline_best.pt")

shutil.copy2(src, dst)
print("Baseline backed up:", dst.exists())
print("Size (MB):", round(dst.stat().st_size / (1024**2), 1))


In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/kaggle/working/yolov7/runs/train/"
    "rdd2022_finetune3/weights/best.pt"
)
backup = Path("/kaggle/working/final_candidate.pt")

if source.exists():
    shutil.copy2(source, backup)
    print("Checkpoint backed up:", backup)
    print("Size:", round(backup.stat().st_size / 1024**2, 1), "MB")
else:
    print("Checkpoint not found. Keep your downloaded model safe.")

In [ ]:
%cd /kaggle/working/yolov7

!python test.py --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --weights /kaggle/working/final_candidate.pt --img 640 --batch 8 --conf 0.001 --iou 0.65 --task val --verbose --name cmp_final

In [ ]:
%cd /kaggle/working/yolov7

!python test.py --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --weights /kaggle/working/improved_road_damage_model.pt --img 640 --batch 8 --conf 0.001 --iou 0.65 --task val --verbose --name cmp_improved

In [ ]:
import sys, torch
sys.path.insert(0, "/kaggle/working/yolov7")   # needed so the model class can be unpickled

src = "/kaggle/working/final_candidate.pt"
dst = "/kaggle/working/app_model.pt"

x = torch.load(src, map_location="cpu", weights_only=False)
if x.get("ema"):
    x["model"] = x["ema"]                      # use the EMA weights if present
for k in ("optimizer", "training_results", "wandb_id", "ema", "updates"):
    x[k] = None
x["epoch"] = -1
x["model"].half()                              # FP16 roughly halves the size
for p in x["model"].parameters():
    p.requires_grad = False
torch.save(x, dst)

In [ ]:
!ls -lh /kaggle/working/app_model.pt

In [ ]:
from PIL import Image
from pathlib import Path
import random, collections

root = Path("/kaggle/working/RDD2022_YOLO/images/train")
print("Folder exists:", root.exists())

files = [p for p in root.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
print("Images found:", len(files))

sizes = collections.defaultdict(collections.Counter)
for p in random.sample(files, min(400, len(files))):
    src = "China_Drone" if p.name.startswith("China_Drone") else "India"
    sizes[src][Image.open(p).size] += 1

for k, v in sizes.items():
    print(k, v.most_common(3))

In [ ]:
from PIL import Image
from pathlib import Path
import random, collections
files = list(Path("/kaggle/working/RDD2022_YOLO/images/train").glob("*.jpg"))
sizes = collections.defaultdict(collections.Counter)
for p in random.sample(files, 400):
    src = "China_Drone" if p.name.startswith("China_Drone") else "India"
    sizes[src][Image.open(p).size] += 1
for k, v in sizes.items(): print(k, v.most_common(3))

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt"):
            p = os.path.join(root, f)
            print(p, round(os.path.getsize(p) / 1024**2, 1), "MB")

In [ ]:
import shutil
shutil.copy2("/kaggle/input/datasets/saksh01/fine-tuned/final_candidate.pt", "/kaggle/working/final_candidate.pt")

In [ ]:
%cd /kaggle/working/yolov7
from pathlib import Path

fixes = {
    "models/experimental.py": ("torch.load(w, map_location=map_location)",
                               "torch.load(w, map_location=map_location, weights_only=False)"),
    "utils/datasets.py": ("torch.load(cache_path)",
                          "torch.load(cache_path, weights_only=False)"),
    "train.py": ("torch.load(weights, map_location=device)",
                 "torch.load(weights, map_location=device, weights_only=False)"),
}
for f, (old, new) in fixes.items():
    p = Path(f)
    s = p.read_text()
    if old in s:
        p.write_text(s.replace(old, new))
        print("Patched", f)
    else:
        print("Already patched or pattern not found:", f)

In [ ]:
from pathlib import Path
import yaml

root = Path("/kaggle/working/RDD2022_YOLO")
for split in ("train", "val", "test"):
    p = root / "images" / split
    print(split, "exists:", p.exists(), "| files:", len(list(p.iterdir())) if p.exists() else 0)

data = {
    "train": str(root / "images/train"),
    "val": str(root / "images/val"),
    "test": str(root / "images/test"),
    "nc": 4,
    "names": ["D00", "D10", "D20", "D40"],
}
with open(root / "rdd2022.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print((root / "rdd2022.yaml").read_text())

In [ ]:
%cd /kaggle/working/yolov7
!python test.py --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --weights /kaggle/working/final_candidate.pt --img 640 --batch 8 --conf 0.001 --iou 0.65 --task val --augment --name cmp_tta

In [ ]:
%cd /kaggle/working/yolov7
import yaml
h = yaml.safe_load(open("data/hyp.scratch.p5.yaml"))
h.update(lr0=0.002, warmup_epochs=1.0, scale=0.5, mixup=0.0, paste_in=0.0)
yaml.safe_dump(h, open("data/hyp.finetune_rdd.yaml", "w"))

In [ ]:
%cd /kaggle/working/yolov7
!python test.py --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --weights /kaggle/working/final_candidate.pt --img 640 --batch 8 --conf 0.001 --iou 0.65 --task val --augment --no-trace --name cmp_tta2

In [ ]:
%cd /kaggle/working/yolov7
!python train.py --workers 2 --device 0 --batch-size 8 --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --img-size 640 640 --cfg cfg/training/yolov7.yaml --weights /kaggle/working/final_candidate.pt --name rdd2022_ft2 --hyp data/hyp.finetune_rdd.yaml --epochs 30

In [ ]:
!tail -n 3 /kaggle/working/yolov7/runs/train/rdd2022_ft2/results.txt
!ls -l --time-style=full-iso /kaggle/working/yolov7/runs/train/rdd2022_ft2/weights/

In [ ]:
import sys, torch
sys.path.insert(0, "/kaggle/working/yolov7")   # needed so the model class can be unpickled

src = "/kaggle/working/yolov7/runs/train/rdd2022_ft2/weights/best.pt"
dst = "/kaggle/working/app_model_v2.pt"

x = torch.load(src, map_location="cpu", weights_only=False)
print("Checkpoint epoch:", x.get("epoch"))
if x.get("ema"):
    x["model"] = x["ema"]
for k in ("optimizer", "training_results", "wandb_id", "ema", "updates"):
    x[k] = None
x["epoch"] = -1
x["model"].half()
for p in x["model"].parameters():
    p.requires_grad = False
torch.save(x, dst)

In [ ]:
!ls -lh /kaggle/working/app_model_v2.pt

In [ ]:
%cd /kaggle/working/yolov7
!python test.py --data /kaggle/working/RDD2022_YOLO/rdd2022.yaml --weights runs/train/rdd2022_ft2/weights/best.pt --img 640 --batch 8 --conf 0.001 --iou 0.65 --task val --verbose --name cmp_ft2